# Diagnóstico de calidad de datos de Instacart

**Actividad:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos
**Rol:** analista de datos de Instacart
**Aprendiz:** jeferson estiven vargas

> **Cómo está construido este notebook:** las cifras no están escritas a mano. Cada resultado se calcula con código a partir de los archivos que se cargan, así las conclusiones siempre corresponden a los datos reales.

## Momento 1. Reconocimiento del proyecto

**Contexto.** Instacart es una plataforma de compras de supermercado en línea. En la EV04 se exploraron sus datos para descubrir patrones de compra. Antes de sacar conclusiones más profundas hay que comprobar si esos datos son confiables.

**Datos disponibles.** El proyecto usa los cinco archivos CSV de Instacart guardados en la carpeta `data/` del repositorio (no se suben a GitHub). En conjunto describen los pedidos, los productos, los productos incluidos en cada pedido, los pasillos y los departamentos de la tienda.

**Objetivo del diagnóstico.** Evaluar la calidad de los datos antes de analizarlos, revisando:

1. La estructura de cada tabla (filas, columnas y tipos de datos).
2. Los valores faltantes.
3. Los tipos de datos incorrectos.
4. Los duplicados y las inconsistencias entre tablas.
5. Las limitaciones y las acciones de limpieza necesarias.

## Momento 2. Pregunta analítica y carga de datos

### Pregunta analítica

> ¿Tienen los cinco conjuntos de datos de Instacart la calidad suficiente (completitud, tipos correctos y consistencia entre tablas) para analizar los patrones de compra de los clientes, y qué acciones de limpieza se necesitan antes de hacerlo?

**Preguntas de apoyo:**

- ¿Qué columnas tienen valores faltantes y en qué proporción?
- ¿Hay columnas cuyo tipo de dato no corresponde a lo que representan?
- ¿Existen filas duplicadas o identificadores que no coinciden entre tablas?

In [ ]:
from pathlib import Path
import csv

import pandas as pd

# El notebook puede ejecutarse desde la carpeta notebooks/ o desde la raíz del repositorio
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Si los CSV ya están en data/raw/ se usan desde ahí; si no, desde data/
CARPETA_RAW = RAIZ / "data" / "raw"
CARPETA_DATOS = CARPETA_RAW if any(CARPETA_RAW.glob("*.csv")) else RAIZ / "data"

print(f"Carpeta de datos: {CARPETA_DATOS}")

In [ ]:
def detectar_separador(ruta: Path) -> str:
    """Lee la primera línea del archivo y detecta el separador de columnas."""
    with open(ruta, encoding="utf-8") as archivo:
        primera_linea = archivo.readline()
    try:
        return csv.Sniffer().sniff(primera_linea, delimiters=",;\t|").delimiter
    except csv.Error:
        return ","


def cargar_csv(ruta: Path) -> pd.DataFrame:
    """Carga un CSV usando el separador detectado."""
    return pd.read_csv(ruta, sep=detectar_separador(ruta))


rutas_csv = sorted(CARPETA_DATOS.glob("*.csv"))
if not rutas_csv:
    raise FileNotFoundError(
        f"No se encontraron archivos .csv en {CARPETA_DATOS}. "
        "Copie los datos de Instacart en la carpeta data/ antes de continuar."
    )

datasets = {}
for ruta in rutas_csv:
    try:
        datasets[ruta.stem] = cargar_csv(ruta)
    except (pd.errors.ParserError, UnicodeDecodeError) as error:
        print(f"No se pudo cargar {ruta.name}: {error}")

print(f"Archivos cargados: {list(datasets)}")

In [ ]:
resumen_carga = pd.DataFrame(
    [
        {
            "archivo": f"{nombre}.csv",
            "separador": repr(detectar_separador(CARPETA_DATOS / f"{nombre}.csv")),
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "tamano_MB": round((CARPETA_DATOS / f"{nombre}.csv").stat().st_size / 1024**2, 2),
        }
        for nombre, df in datasets.items()
    ]
)
resumen_carga

### Interpretación de la carga

In [ ]:
total_filas = resumen_carga["filas"].sum()
mayor = resumen_carga.loc[resumen_carga["filas"].idxmax()]
menor = resumen_carga.loc[resumen_carga["filas"].idxmin()]

print(f"Se cargaron {len(datasets)} archivos con {total_filas:,} filas en total.")
print(f"La tabla más grande es {mayor['archivo']} ({mayor['filas']:,} filas) "
      f"y la más pequeña es {menor['archivo']} ({menor['filas']:,} filas).")
print("OK: se cargaron los 5 archivos esperados." if len(datasets) == 5
      else f"ATENCIÓN: se esperaban 5 archivos y se cargaron {len(datasets)}.")

## Momento 3. Inspección estructural de los datasets

En este momento se revisa la estructura de cada tabla con los métodos básicos de pandas:

| Método | Qué muestra |
|---|---|
| `head()` / `tail()` | Las primeras y las últimas filas, para ver cómo lucen los datos reales |
| `shape` | Cuántas filas y columnas tiene la tabla |
| `columns` | Los nombres de las columnas |
| `dtypes` | El tipo de dato de cada columna |
| `info()` | Resumen de tipos, valores no nulos y memoria usada |
| `describe()` | Estadísticas descriptivas (conteo, media, mínimo, máximo, etc.) |

In [ ]:
from IPython.display import Markdown, display

for nombre, df in datasets.items():
    display(Markdown(f"### `{nombre}` · primeras y últimas 3 filas"))
    display(pd.concat([df.head(3), df.tail(3)]))

### Dimensiones, columnas y tipos de datos

In [ ]:
for nombre, df in datasets.items():
    print(f"{nombre}: shape = {df.shape}")
    print(f"  columnas: {list(df.columns)}")
    print(f"  dtypes:   {df.dtypes.astype(str).to_dict()}")
    print()

In [ ]:
tipos_por_columna = pd.DataFrame(
    [
        {"dataset": nombre, "columna": columna, "tipo": str(tipo)}
        for nombre, df in datasets.items()
        for columna, tipo in df.dtypes.items()
    ]
)
tipos_por_columna

### Resumen con `info()`

In [ ]:
for nombre, df in datasets.items():
    display(Markdown(f"#### `{nombre}`"))
    df.info(memory_usage="deep")

### Estadísticas descriptivas con `describe()`

In [ ]:
for nombre, df in datasets.items():
    display(Markdown(f"#### `{nombre}`"))
    if not df.select_dtypes(include="number").empty:
        display(df.describe().T)
    if not df.select_dtypes(include="object").empty:
        display(df.describe(include="object").T)

### Tabla del diagnóstico estructural

In [ ]:
diagnostico_estructural = pd.DataFrame(
    [
        {
            "dataset": nombre,
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "columnas_numericas": df.select_dtypes(include="number").shape[1],
            "columnas_texto": df.select_dtypes(include="object").shape[1],
            "memoria_MB": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        }
        for nombre, df in datasets.items()
    ]
)
diagnostico_estructural

### Interpretación de la inspección

In [ ]:
mas_memoria = diagnostico_estructural.loc[diagnostico_estructural["memoria_MB"].idxmax()]
print(f"La tabla que más memoria usa es '{mas_memoria['dataset']}' "
      f"con {mas_memoria['memoria_MB']} MB.")

# Columnas que deberían ser enteras (identificadores y contadores) pero quedaron como decimales
columnas_float = tipos_por_columna[tipos_por_columna["tipo"].str.startswith("float")]
if columnas_float.empty:
    print("Ninguna columna quedó como decimal (float).")
else:
    print("Columnas cargadas como decimales (float); suele indicar valores faltantes, se revisa en el Momento 4:")
    for _, fila in columnas_float.iterrows():
        print(f"  - {fila['dataset']}.{fila['columna']}")

columnas_texto = tipos_por_columna[tipos_por_columna["tipo"].isin(["object", "str", "string"])]
print(f"Hay {len(columnas_texto)} columnas de texto: "
      f"{[f'{d}.{c}' for d, c in zip(columnas_texto['dataset'], columnas_texto['columna'])]}")